
# 02 — Preprocessing Axe 3 (Version 1)

## Objectif de ce notebook

Ce notebook prépare les données du dataset IST pour la **modélisation de l’Axe 3**, c’est-à-dire la **prédiction de la mortalité post-AVC** :

- `DDEAD` : mortalité à **14 jours**
- `FDEAD` : mortalité à **6 mois**

L’objectif ici n’est **pas encore d’entraîner les modèles**, mais de construire une base de données **propre, cohérente et exploitable** pour la phase de modeling.

---

## Ce que contient cette Version 1

Dans cette première version, on met en place un preprocessing **simple, lisible et robuste** :

1. importation des librairies ;
2. chargement du dataset depuis Google Drive ;
3. sélection des variables pertinentes pour l’Axe 3 ;
4. nettoyage des colonnes et standardisation des valeurs manquantes ;
5. préparation des cibles `DDEAD` et `FDEAD` ;
6. séparation des variables explicatives et des cibles ;
7. encodage des variables catégorielles ;
8. imputation des valeurs manquantes ;
9. sauvegarde des fichiers prêts pour la modélisation.

---

## Rappel méthodologique

Pour l’Axe 3, on cherche à prédire la mortalité à partir des informations **utiles cliniquement** et **disponibles au bon moment**.  
On évite donc d’introduire dans les features des variables qui fuient l’information cible ou qui correspondent déjà à des outcomes tardifs.

Cette version 1 reste volontairement **simple** :
- pas de feature engineering avancé ;
- pas encore de gestion sophistiquée du déséquilibre ;
- pas encore d’optimisation des variables.

Ces améliorations seront réservées à la **Version 2**.



## Rappel du cahier des charges

Le cahier des charges précise que l’**Axe 3** porte sur la **prédiction de la mortalité à 14 jours (`DDEAD`) et à 6 mois (`FDEAD`)** à partir du dataset **IST corrected.csv**.  
Il indique également que, pour cet axe, les métriques importantes sont notamment **l’AUC-ROC** et surtout le **Recall**, afin de limiter les faux négatifs cliniquement dangereux.


In [1]:

# =========================
# 1. Imports
# =========================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from google.colab import drive
from sklearn.impute import SimpleImputer

# Affichage plus confortable
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)



## Chargement du dataset depuis Google Drive

**Important :**
- le dataset IST peut provoquer une erreur d’encodage en UTF-8, donc on utilise ici `encoding="latin1"`.


In [2]:

# =========================
# 2. Chargement du dataset
# =========================

drive.mount('/content/drive')

path = "/content/drive/MyDrive/Stroke_Project_ML/Data/IST_corrected.csv"

df = pd.read_csv(path, encoding="latin1")

print("Dataset chargé avec succès.")
print(f"Nombre de lignes : {df.shape[0]}")
print(f"Nombre de colonnes : {df.shape[1]}")

df.head()


Mounted at /content/drive
Dataset chargé avec succès.
Nombre de lignes : 19435
Nombre de colonnes : 112


,HOSPNUM,RDELAY,RCONSC,SEX,AGE,RSLEEP,RATRIAL,RCT,RVISINF,RHEP24,RASP3,RSBP,RDEF1,RDEF2,RDEF3,RDEF4,RDEF5,RDEF6,RDEF7,RDEF8,STYPE,RDATE,HOURLOCAL,MINLOCAL,DAYLOCAL,RXASP,RXHEP,DASP14,DASPLT,DLH14,DMH14,DHH14,ONDRUG,DSCH,DIVH,DAP,DOAC,DGORM,DSTER,DCAA,DHAEMD,DCAREND,DTHROMB,DMAJNCH,DMAJNCHD,DMAJNCHX,DSIDE,DSIDED,DSIDEX,DDIAGISC,DDIAGHA,DDIAGUN,DNOSTRK,DNOSTRKX,DRSISC,DRSISCD,DRSH,DRSHD,DRSUNK,DRSUNKD,DPE,DPED,DALIVE,DALIVED,DPLACE,DDEAD,DDEADD,DDEADC,DDEADX,FDEAD,FLASTD,FDEADD,FDEADC,FDEADX,FRECOVER,FDENNIS,FPLACE,FAP,FOAC,FU1_RECD,FU2_DONE,COUNTRY,CNTRYNUM,FU1_COMP,NCCODE,CMPLASP,CMPLHEP,DIED,TD,EXPDD,EXPD6,EXPD14,SET14D,ID14,OCCODE,DEAD1,DEAD2,DEAD3,DEAD4,DEAD5,DEAD6,DEAD7,DEAD8,H14,ISC14,NK14,STRK14,HTI14,PE14,DVT14,TRAN14,NCB14
0,1,17,D,M,69,Y,NaN,Y,Y,NaN,NaN,140,N,N,N,Y,N,Y,N,Y,PACS,sty-91,99,99,4,Y,N,Y,Y,N,NaN,N,14.0,NaN,NaN,N,N,N,N,N,N,NaN,NaN,N,NaN,NaN,N,NaN,NaN,Y,N,N,N,NaN,N,NaN,N,NaN,NaN,NaN,N,NaN,N,NaN,NaN,N,NaN,0.0,NaN,N,NaN,NaN,NaN,NaN,N,Y,E,NaN,NaN,14.0,187.0,UK,27,NaN,13,Y,Y,0,187.0,0.6980,0.2344,0.1054,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,10,F,M,76,Y,NaN,Y,N,NaN,NaN,150,Y,Y,Y,N,N,N,N,N,LACS,sty-91,99,99,7,N,L,N,Y,Y,NaN,N,14.0,NaN,NaN,N,N,N,N,Y,N,NaN,NaN,N,NaN,NaN,N,NaN,NaN,Y,N,N,N,NaN,N,NaN,N,NaN,NaN,NaN,N,NaN,N,NaN,NaN,N,NaN,0.0,NaN,N,NaN,NaN,NaN,NaN,N,Y,A,NaN,NaN,16.0,192.0,UK,27,NaN,NaN,Y,Y,0,192.0,0.5389,0.1555,0.0421,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,43,F,F,71,N,NaN,Y,N,NaN,NaN,170,Y,Y,Y,N,N,N,N,N,LACS,sty-91,99,99,3,Y,N,Y,Y,N,NaN,N,11.0,NaN,NaN,N,N,N,N,N,N,NaN,NaN,N,NaN,NaN,N,NaN,NaN,Y,N,N,N,NaN,N,NaN,N,NaN,NaN,NaN,N,NaN,Y,11.0,NaN,N,NaN,0.0,NaN,N,NaN,NaN,NaN,NaN,N,Y,A,NaN,NaN,11.0,189.0,UK,27,NaN,NaN,Y,Y,0,189.0,0.5275,0.1009,0.0323,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1,6,F,M,81,N,NaN,N,N,NaN,NaN,170,N,N,N,Y,N,N,N,N,PACS,sty-91,99,99,7,N,H,N,Y,N,NaN,Y,14.0,NaN,NaN,N,N,N,N,Y,N,NaN,NaN,N,NaN,NaN,N,NaN,NaN,Y,N,N,N,NaN,N,NaN,N,NaN,NaN,NaN,N,NaN,Y,16.0,NaN,N,NaN,0.0,NaN,N,NaN,NaN,NaN,NaN,Y,N,A,NaN,NaN,23.0,183.0,UK,27,NaN,NaN,Y,Y,0,183.0,0.4021,0.1147,0.0244,1,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,4,20,F,M,78,N,NaN,N,N,NaN,NaN,170,Y,Y,Y,N,N,N,N,N,LACS,lut-91,99,99,6,Y,H,Y,Y,N,NaN,Y,14.0,NaN,NaN,N,N,N,N,N,N,NaN,NaN,N,NaN,NaN,N,NaN,NaN,Y,N,N,N,NaN,N,NaN,N,NaN,NaN,NaN,N,NaN,N,NaN,NaN,N,NaN,0.0,NaN,N,NaN,NaN,NaN,NaN,N,Y,E,NaN,NaN,17.0,214.0,UK,27,NaN,NaN,Y,Y,0,214.0,0.5600,0.1709,0.0441,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0



## Première vérification de structure

Avant tout preprocessing, il faut vérifier que :
- les colonnes sont bien présentes ;
- les variables cibles existent ;
- le dataset correspond bien à la structure attendue.

Cette étape est utile pour éviter les erreurs silencieuses plus loin dans le pipeline.


In [3]:

# =========================
# 3. Vérification générale
# =========================

print("Colonnes disponibles :")
display(pd.DataFrame({"colonne": df.columns}))

print("\nVérification des cibles :")
print("DDEAD présente ?", "DDEAD" in df.columns)
print("FDEAD présente ?", "FDEAD" in df.columns)


Colonnes disponibles :


,colonne
0,HOSPNUM
1,RDELAY
2,RCONSC
3,SEX
4,AGE
5,RSLEEP
6,RATRIAL
7,RCT
8,RVISINF
9,RHEP24



Vérification des cibles :
DDEAD présente ? True
FDEAD présente ? True



## Sélection des variables pour l’Axe 3

Pour cette **Version 1**, on conserve un sous-ensemble de variables simples et cohérentes avec l’objectif clinique :

### Variables d’entrée retenues
- **Admission / baseline** : `AGE`, `SEX`, `RCONSC`, `RSBP`, `RATRIAL`, `RDELAY`
- **Déficits neurologiques** : `RDEF1` à `RDEF8`
- **Type d’AVC** : `STYPE`
- **Traitement précoce** : `RXASP`, `RXHEP`

### Variables cibles
- `DDEAD` : mortalité à 14 jours
- `FDEAD` : mortalité à 6 mois

### Pourquoi ce choix ?
On cherche ici à construire une base **simple et défendable** pour une première modélisation, sans inclure des variables trop tardives qui pourraient provoquer du **data leakage**.


In [4]:

# =========================
# 4. Sélection des variables
# =========================

selected_columns = [
    "AGE", "SEX", "RCONSC", "RSBP", "RATRIAL", "RDELAY",
    "RDEF1", "RDEF2", "RDEF3", "RDEF4", "RDEF5", "RDEF6", "RDEF7", "RDEF8",
    "STYPE", "RXASP", "RXHEP",
    "DDEAD", "FDEAD"
]

# On garde uniquement les colonnes utiles
df_model = df[selected_columns].copy()

print("Shape après sélection :", df_model.shape)
df_model.head()


Shape après sélection : (19435, 19)


,AGE,SEX,RCONSC,RSBP,RATRIAL,RDELAY,RDEF1,RDEF2,RDEF3,RDEF4,RDEF5,RDEF6,RDEF7,RDEF8,STYPE,RXASP,RXHEP,DDEAD,FDEAD
0,69,M,D,140,NaN,17,N,N,N,Y,N,Y,N,Y,PACS,Y,N,N,N
1,76,M,F,150,NaN,10,Y,Y,Y,N,N,N,N,N,LACS,N,L,N,N
2,71,F,F,170,NaN,43,Y,Y,Y,N,N,N,N,N,LACS,Y,N,N,N
3,81,M,F,170,NaN,6,N,N,N,Y,N,N,N,N,PACS,N,H,N,N
4,78,M,F,170,NaN,20,Y,Y,Y,N,N,N,N,N,LACS,Y,H,N,N



## Nettoyage général du dataset

Avant de traiter séparément `DDEAD` et `FDEAD`, on effectue un nettoyage de base :

1. suppression des espaces superflus dans les noms de colonnes ;
2. standardisation de certains faux codes de valeurs manquantes (`?`, chaîne vide, etc.) ;
3. conversion de certaines colonnes numériques si nécessaire.

L’idée est de rendre les données plus homogènes avant les étapes de préparation ciblée.


In [5]:

# =========================
# 5. Nettoyage général
# =========================

# Nettoyage des noms de colonnes
df_model.columns = df_model.columns.str.strip()

# Standardisation de quelques faux codes de valeurs manquantes
df_model = df_model.replace(["?", "??", "NA", "N/A", ""], np.nan)

print("Aperçu des valeurs manquantes :")
display(df_model.isna().sum().sort_values(ascending=False).to_frame("nb_valeurs_manquantes"))


Aperçu des valeurs manquantes :


,nb_valeurs_manquantes
RATRIAL,984
FDEAD,99
DDEAD,19
RCONSC,0
SEX,0
AGE,0
RSBP,0
RDEF2,0
RDEF3,0
RDELAY,0



## Préparation de la cible `DDEAD` (mortalité à 14 jours)

Dans le dataset IST, la variable `DDEAD` est codée de manière catégorielle :
- `Y` : décès
- `N` : non décès
- `U` : inconnu

Pour pouvoir entraîner un modèle supervisé, il faut convertir cette cible en format binaire :
- `Y` → `1`
- `N` → `0`
- `U` / valeurs inconnues → `NaN`

Ensuite, on supprime uniquement les lignes où la **cible** est inconnue.


In [6]:

# =========================
# 6. Préparation de la cible DDEAD
# =========================

# Copie de travail pour DDEAD
df_ddead = df_model.copy()

# Uniformisation du format texte
df_ddead["DDEAD"] = df_ddead["DDEAD"].astype(str).str.strip().str.upper()

# Mapping des modalités vers une cible binaire
mapping_ddead = {
    "Y": 1,
    "N": 0,
    "U": np.nan,
    "NAN": np.nan,
    "": np.nan
}

df_ddead["DDEAD"] = df_ddead["DDEAD"].replace(mapping_ddead)
df_ddead["DDEAD"] = pd.to_numeric(df_ddead["DDEAD"], errors="coerce")

print("Distribution DDEAD après mapping :")
print(df_ddead["DDEAD"].value_counts(dropna=False))

# Suppression des lignes dont la cible est inconnue
df_ddead = df_ddead.dropna(subset=["DDEAD"]).copy()

print("\nShape df_ddead après suppression des cibles manquantes :", df_ddead.shape)


Distribution DDEAD après mapping :
DDEAD
0.0    17376
1.0     2034
NaN       25
Name: count, dtype: int64

Shape df_ddead après suppression des cibles manquantes : (19410, 19)



## Préparation de la cible `FDEAD` (mortalité à 6 mois)

On applique exactement la même logique à `FDEAD` :

- `Y` → `1`
- `N` → `0`
- `U` / valeurs inconnues → `NaN`

Cela permet d’obtenir un second dataset propre pour la prédiction de la mortalité à 6 mois.


In [7]:

# =========================
# 7. Préparation de la cible FDEAD
# =========================

# Copie de travail pour FDEAD
df_fdead = df_model.copy()

# Uniformisation du format texte
df_fdead["FDEAD"] = df_fdead["FDEAD"].astype(str).str.strip().str.upper()

# Mapping des modalités vers une cible binaire
mapping_fdead = {
    "Y": 1,
    "N": 0,
    "U": np.nan,
    "NAN": np.nan,
    "": np.nan
}

df_fdead["FDEAD"] = df_fdead["FDEAD"].replace(mapping_fdead)
df_fdead["FDEAD"] = pd.to_numeric(df_fdead["FDEAD"], errors="coerce")

print("Distribution FDEAD après mapping :")
print(df_fdead["FDEAD"].value_counts(dropna=False))

# Suppression des lignes dont la cible est inconnue
df_fdead = df_fdead.dropna(subset=["FDEAD"]).copy()

print("\nShape df_fdead après suppression des cibles manquantes :", df_fdead.shape)


Distribution FDEAD après mapping :
FDEAD
0.0    14910
1.0     4369
NaN      156
Name: count, dtype: int64

Shape df_fdead après suppression des cibles manquantes : (19279, 19)



## Construction des jeux `X` et `y`

À ce stade, on prépare séparément les données pour les deux tâches :

- **Task 1 :** prédire `DDEAD`
- **Task 2 :** prédire `FDEAD`

On retire des features :
- la cible à prédire ;
- l’autre cible, afin d’éviter d’utiliser un outcome comme variable explicative.

C’est une précaution importante pour éviter le **data leakage**.


In [8]:

# =========================
# 8. Séparation Features / Target
# =========================

# Jeu pour DDEAD
X_ddead = df_ddead.drop(columns=["DDEAD", "FDEAD"], errors="ignore")
y_ddead = df_ddead["DDEAD"].astype(int)

# Jeu pour FDEAD
X_fdead = df_fdead.drop(columns=["FDEAD", "DDEAD"], errors="ignore")
y_fdead = df_fdead["FDEAD"].astype(int)

print("Shape X_ddead :", X_ddead.shape)
print("Shape y_ddead :", y_ddead.shape)
print()
print("Shape X_fdead :", X_fdead.shape)
print("Shape y_fdead :", y_fdead.shape)


Shape X_ddead : (19410, 17)
Shape y_ddead : (19410,)

Shape X_fdead : (19279, 17)
Shape y_fdead : (19279,)



## Identification des variables catégorielles et numériques

Avant l’encodage et l’imputation, on distingue :
- les variables **catégorielles** (`object`) ;
- les variables **numériques** (`int64`, `float64`).

Cette séparation permet d’appliquer des traitements adaptés :
- **mode** pour les catégorielles ;
- **médiane** pour les numériques.


In [9]:

# =========================
# 9. Identification des types de variables
# =========================

cat_cols = X_ddead.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_ddead.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Variables catégorielles :")
print(cat_cols)

print("\nVariables numériques :")
print(num_cols)


Variables catégorielles :
['SEX', 'RCONSC', 'RATRIAL', 'RDEF1', 'RDEF2', 'RDEF3', 'RDEF4', 'RDEF5', 'RDEF6', 'RDEF7', 'RDEF8', 'STYPE', 'RXASP', 'RXHEP']

Variables numériques :
['AGE', 'RSBP', 'RDELAY']



## Imputation des valeurs manquantes

Dans cette **Version 1**, on utilise une stratégie simple et classique :

- **médiane** pour les variables numériques ;
- **modalité la plus fréquente** pour les variables catégorielles.

Cette approche est adaptée à une première version propre du pipeline.


In [10]:

# =========================
# 10. Imputation des valeurs manquantes
# =========================

# Imputers
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

# Application sur X_ddead
if len(num_cols) > 0:
    X_ddead[num_cols] = num_imputer.fit_transform(X_ddead[num_cols])
if len(cat_cols) > 0:
    X_ddead[cat_cols] = cat_imputer.fit_transform(X_ddead[cat_cols])

# Application sur X_fdead
if len(num_cols) > 0:
    X_fdead[num_cols] = num_imputer.fit_transform(X_fdead[num_cols])
if len(cat_cols) > 0:
    X_fdead[cat_cols] = cat_imputer.fit_transform(X_fdead[cat_cols])

print("NaN restants dans X_ddead :", X_ddead.isna().sum().sum())
print("NaN restants dans X_fdead :", X_fdead.isna().sum().sum())


NaN restants dans X_ddead : 0
NaN restants dans X_fdead : 0



## Encodage des variables catégorielles

Les modèles de machine learning ne peuvent pas exploiter directement les variables textuelles.

On applique donc un **One-Hot Encoding** avec `pd.get_dummies()` pour transformer les catégories en variables numériques.

Dans cette version 1 :
- on garde un encodage simple ;
- on utilise `drop_first=True` pour limiter un peu la redondance.


In [11]:

# =========================
# 11. Encodage des variables catégorielles
# =========================

X_ddead_encoded = pd.get_dummies(X_ddead, columns=cat_cols, drop_first=True)
X_fdead_encoded = pd.get_dummies(X_fdead, columns=cat_cols, drop_first=True)

print("Shape X_ddead après encodage :", X_ddead_encoded.shape)
print("Shape X_fdead après encodage :", X_fdead_encoded.shape)


Shape X_ddead après encodage : (19410, 31)
Shape X_fdead après encodage : (19279, 31)



## Vérification finale avant sauvegarde

Avant de passer au notebook de modélisation, il faut vérifier que :

- les jeux de données ne contiennent plus de valeurs manquantes ;
- les shapes sont cohérentes ;
- les targets sont bien binaires.


In [12]:

# =========================
# 12. Vérifications finales
# =========================

print("NaN restants dans X_ddead_encoded :", X_ddead_encoded.isna().sum().sum())
print("NaN restants dans X_fdead_encoded :", X_fdead_encoded.isna().sum().sum())

print("\nDistribution finale y_ddead :")
print(y_ddead.value_counts())

print("\nDistribution finale y_fdead :")
print(y_fdead.value_counts())


NaN restants dans X_ddead_encoded : 0
NaN restants dans X_fdead_encoded : 0

Distribution finale y_ddead :
DDEAD
0    17376
1     2034
Name: count, dtype: int64

Distribution finale y_fdead :
FDEAD
0    14910
1     4369
Name: count, dtype: int64



## Sauvegarde des fichiers preprocessés

On exporte maintenant les jeux préparés dans Google Drive pour pouvoir les réutiliser facilement dans le notebook de modeling.

### Fichiers sauvegardés
- `X_ddead_preprocessed_v1.csv`
- `y_ddead_preprocessed_v1.csv`
- `X_fdead_preprocessed_v1.csv`
- `y_fdead_preprocessed_v1.csv`


In [13]:

# =========================
# 13. Sauvegarde des fichiers
# =========================

save_dir = "/content/drive/MyDrive/Stroke_Project_ML/Data/"

X_ddead_encoded.to_csv(save_dir + "X_ddead_preprocessed_v1.csv", index=False)
y_ddead.to_csv(save_dir + "y_ddead_preprocessed_v1.csv", index=False)

X_fdead_encoded.to_csv(save_dir + "X_fdead_preprocessed_v1.csv", index=False)
y_fdead.to_csv(save_dir + "y_fdead_preprocessed_v1.csv", index=False)

print("Fichiers sauvegardés avec succès dans Google Drive.")


Fichiers sauvegardés avec succès dans Google Drive.



## Conclusion

Cette **Version 1 du preprocessing** fournit une base propre pour passer à la modélisation de l’Axe 3.

### Ce qui a été fait
- chargement robuste du dataset IST dans Colab ;
- sélection des variables utiles ;
- nettoyage des cibles `DDEAD` et `FDEAD` ;
- suppression des cibles inconnues ;
- imputation simple des valeurs manquantes ;
- encodage des variables catégorielles ;
- export des jeux prêts pour le modeling.

### Ce qui sera amélioré en Version 2
- sélection plus fine des variables ;
- feature engineering clinique ;
- gestion plus avancée du déséquilibre ;
- pipeline sklearn plus rigoureux ;
- préparation encore plus robuste pour la comparaison de modèles.
